# 모듈 1: 블랙파우더 성분 데이터 전처리 - EDA 및 검증

`src/pipeline.py`의 각 단계를 순서대로 호출해 원본 데이터를 정제하고, 전/후 비교 시각화와 파생변수·원료등급 기여도(ablation test)를 확인한다.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from pathlib import Path

from src.preprocessing import load_raw, clean_schema
from src.imputation import flag_outliers, impute_missing
from src.features import engineer_features
from src.validation import (
    check_ranges,
    plot_missing_before_after,
    plot_distributions,
    plot_correlation_heatmap,
    run_ablation_test,
)
from src.config import COMPOSITION_COLUMNS, PROCESS_COLUMNS, TARGET_COLUMNS

RAW_PATH = Path('../data/raw/영풍그룹_금속회수율최적화_RawMaterialData_학생배포용.xlsx')
FIGURES_DIR = Path('../reports/figures')

## 1. 원본 로드 및 정제

In [ ]:
raw_df = load_raw(RAW_PATH)
clean_df = clean_schema(raw_df)
outlier_flagged_df = flag_outliers(clean_df)
imputed_df = impute_missing(outlier_flagged_df)
final_df = engineer_features(imputed_df)
final_df.head()

## 2. 결측치 처리 전/후 비교

In [ ]:
plot_missing_before_after(outlier_flagged_df, imputed_df, FIGURES_DIR)

## 3. 이상치 제거 전/후 분포

In [ ]:
plot_distributions(raw_df, ['Ni_함량_pct', '황산투입량_L', '침출온도_C'], FIGURES_DIR / 'before')
plot_distributions(final_df, ['Ni_함량_pct', '황산투입량_L', '침출온도_C'], FIGURES_DIR / 'after')

## 4. 상관관계 히트맵

In [ ]:
feature_columns = COMPOSITION_COLUMNS + PROCESS_COLUMNS + [
    'acid_to_ore_ratio', 'leach_intensity', 'total_impurity_pct', 'target_purity_ratio'
]
plot_correlation_heatmap(final_df, feature_columns, TARGET_COLUMNS, FIGURES_DIR)

## 5. 정합성 검증

In [ ]:
violations = check_ranges(final_df)
print('정상범위 위반:', violations if violations else '없음')

## 6. 파생변수 및 원료등급 기여도 검증 (Ablation Test)

In [ ]:
base_features = COMPOSITION_COLUMNS + PROCESS_COLUMNS
# analysis_month는 회수율 예측 후보 변수가 아니라 계절성/설비 드리프트 모니터링용 캘린더 파생변수이므로, 특성 기여도 비교(히트맵·ablation test) 대상에서 제외한다.
derived_features = ['acid_to_ore_ratio', 'leach_intensity', 'total_impurity_pct', 'target_purity_ratio']

feature_sets = {
    'base_only': base_features,
    'base_plus_grade': base_features + ['원료등급'],
    'base_plus_derived': base_features + derived_features,
    'all_features': base_features + ['원료등급'] + derived_features,
}

results = run_ablation_test(final_df, feature_sets, TARGET_COLUMNS)
pd.DataFrame(results).T

## 7. 심층 인터뷰 답변

**1) 실제 공정 데이터 품질 문제를 어떻게 식별·처리했는가?**

데이터사전에 정의된 정상 범위를 벗어나는 값(`Ni_함량_pct=999.0`, `침출온도_C=-5.0`, `황산투입량_L=15000`)을 물리적으로 불가능한 이상치로 식별해 `flag_outliers`에서 NaN으로 치환했다. 이후 남은 랜덤 결측(조성 6~9%, 공정조건)은 변수 성격에 따라 원료등급별 중앙값, 회귀 기반, 배치ID 순서 기반 보간으로 구분 처리했다 (설계 문서 §4).

**2) 특성공학 과정에서 도메인 지식을 어떻게 활용했는가?**

산-광석비, 침출강도(온도×시간), 총불순물, 목표금속 순도비는 모두 습식제련 침출 공정의 화학양론·반응속도론적 지식에서 도출했다 (설계 문서 §6). 위 ablation test로 파생변수·원료등급 포함 시 R²/RMSE가 실제로 개선되는지 정량 확인했다.

**3) 데이터 불균형 문제가 있었다면 어떻게 해결했는가?**

전체 424행 기준 `원료등급`(A/B/C) 분포는 B 38.9%, A 35.1%, C 25.9%로(아래 셀 실행 결과), 가장 적은 클래스(C)도 약 26%를 차지해 특정 클래스가 극단적으로 희소한 심각한 불균형은 아니라고 판단했다. 설계 문서 §9에서는 구현 단계의 실제 분포를 보고 불균형 여부와 계층적 샘플링(stratified split) 적용 여부를 결정하기로 했는데, 실측 결과 클래스 간 비율 차이가 26~39% 수준으로 완만해 SMOTE·계층적 샘플링·클래스 가중치 등 별도의 불균형 보정은 적용하지 않기로 결론지었다. 다만 위 ablation test 결과를 보면 `base_only`(Ni R²=0.580, Co R²=0.601) 대비 `base_plus_grade`(Ni R²=0.581, Co R²=0.602)는 R²가 약 0.001 수준만 상승해 `원료등급`을 모델 입력 특성으로 그대로 추가하는 것 자체는 회귀 성능에 유의미한 영향을 주지 않는다. 그럼에도 `원료등급`은 Task 5의 등급별 중앙값 보간(grade-based median imputation)에서 그룹핑 키로 쓰이므로 컬럼 자체는 계속 유지한다. 반면 산-광석비·침출강도·총불순물·목표금속 순도비 등 파생변수는 `base_plus_derived`에서 R²를 Ni 0.580→0.811, Co 0.601→0.793 수준으로 크게 끌어올려, 현재 데이터에서 회귀 성능에 실질적으로 기여하는 것은 `원료등급`이 아니라 파생변수 쪽임을 ablation test가 보여준다.

In [ ]:
final_df['원료등급'].value_counts(normalize=True) * 100